# Workflow: presentation figures

This notebook regenerates every figure in `presentation_figures` (01–17) for the DRD2 antagonist project
from the results of the earlier workflow steps (ChEMBL curation, QSAR models, virtual screening,
docking with AutoDock Vina and the MD simulation of Hit 01).

**Inputs** are read from the group project folder `ACMDD_D2_Receptor` (set `PROJECT` below).
**Outputs** are written to `results/presentation_figures/` in this repository.

| # | Figure | Source data |
|---|--------|-------------|
| 01 | Training set pIC50 distribution | `data/DopamineD2_compounds.csv` |
| 02 | Classification benchmark | `results/ml_results_crossvalidation.csv` |
| 03 | Regression benchmark | `results/reg_results_crossvalidation.csv` |
| 04 | QSAR predicted vs. measured | `data/DopamineD2_morgan2.csv` (SVR, 5-fold CV) |
| 05 | Virtual screening pIC50 | random ZINC + scaffold-library predictions |
| 06 | ML vs. docking (top-100) | `Workflow_steps/data/top100_substructure_filters.csv` |
| 07 | Docking scores vs. controls | top-100 docking runs + control runs in `docking/runs` |
| 08 | Inactive vs. active state | `results/highest_potential` + dopamine state comparison |
| 09 | Top-5 hit structures | `results/highest_potential/state_comparison_summary.csv` |
| 10 | Cluster 4 docking vs. pIC50 | `Workflow_steps/data/cluster4_docking_results.csv` |
| 11 | Docking seed reproducibility | `docking/runs/ligand_1_supplied_6CM4/docking_summary.csv` |
| 12 | MD of Hit 01 | `Workflow_steps/data/topology.pdb`, `trajectory.xtc`, `simulation_log.txt` |
| 13 | Hit poses, inactive vs. active (PyMOL) | `results/highest_potential/<hit>/<state>/` |
| 14 | Risperidone redocking (PyMOL) | `docking/runs/Risperidone` |
| 15 | Top-5 overlay (PyMOL) | `results/highest_potential/<hit>/6CM4/` |
| 16 | DRD2 overview (PyMOL) | `Naked_6CM4_NA_lig1.mmcif` |
| 17 | MD residue–ligand distances | `MD_residue_ligand_distances.csv` |

Figures 12 and 13–16 need `MDAnalysis` and `pymol-open-source`
(`conda install -c conda-forge mdanalysis pymol-open-source`).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import pearsonr

from sklearn.svm import SVR
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error

from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdMolAlign

In [ ]:
HERE = Path(_dh[-1])

# Group project folder with all workflow outputs (change this if it lives elsewhere)
PROJECT = Path.home() / "Desktop" / "ACMDD group 6 assignment" / "ACMDD_D2_Receptor"
DATA = PROJECT / "Workflow_steps" / "data"
RUNS = PROJECT / "docking" / "runs"
HIT_DIR = PROJECT / "results" / "highest_potential"

FIG_DIR = HERE.parent / "results" / "presentation_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not PROJECT.is_dir():
    raise FileNotFoundError(f"Project folder not found: {PROJECT}")

## Shared style

One colour per role, used in every figure: blue = active / inactive-state receptor / scaffold library,
orange = selected hits / active-state receptor, green = ANN, grey = inactive / random library.

In [ ]:
BLUE, ORANGE, GREEN, GREY = "#2A78D6", "#EB6834", "#1BAF7A", "#A3A29D"
HIT_COLORS = [ORANGE, BLUE, GREEN, "#E377A0", "#5B4BD6"]
ACTIVITY_CUTOFF = 6.3  # pIC50 cut-off used for the active/inactive labels

plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 17,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#555555",
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#E3E3E0",
    "xtick.color": "#555555",
    "ytick.color": "#555555",
    "legend.frameon": False,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})


def save(fig, filename):
    """Save a figure to FIG_DIR and show it in the notebook."""
    fig.savefig(FIG_DIR / filename)
    plt.show()


def best_vina_score(pdbqt_file):
    """Return the score of the top-ranked pose in a Vina output PDBQT file."""
    with open(pdbqt_file) as f:
        for line in f:
            if line.startswith("REMARK VINA RESULT"):
                return float(line.split()[3])
    raise ValueError(f"No Vina result in {pdbqt_file}")

## Load the shared tables

The five selected hits come from the inactive (6CM4) vs. active (9BS9) state comparison. We add their
ZINC library ID, predicted pIC50 and top-100 Vina score from the top-100 selection table (matched on SMILES).

In [ ]:
training = pd.read_csv(PROJECT / "data" / "DopamineD2_compounds.csv")
top100 = pd.read_csv(DATA / "top100_substructure_filters.csv")
selection = pd.read_csv(DATA / "top100_selection_table.csv")

hits = pd.read_csv(HIT_DIR / "state_comparison_summary.csv").merge(
    selection[["smiles", "compound_id", "pred_pIC50", "vina_score"]].rename(
        columns={"compound_id": "library_id", "vina_score": "vina_top100"}
    ),
    on="smiles",
)
hits["label"] = hits["compound_id"].str.replace("_", " ")
hits[["label", "library_id", "pred_pIC50", "vina_score_6CM4_kcal_mol", "vina_score_9BS9_kcal_mol"]]

## Figure 01 – Training set pIC50 distribution

In [ ]:
is_active = training["pIC50"] >= ACTIVITY_CUTOFF
bins = np.arange(3.5, 10.51, 0.25)

fig, ax = plt.subplots(figsize=(10, 5.2))
ax.hist(
    [training.loc[~is_active, "pIC50"], training.loc[is_active, "pIC50"]],
    bins=bins,
    stacked=True,
    color=[GREY, BLUE],
    edgecolor="white",
    label=[f"Inactive (n = {(~is_active).sum()})", f"Active (n = {is_active.sum()})"],
)
ax.axvline(ACTIVITY_CUTOFF, color="black", ls="--", lw=1.5)
ax.text(ACTIVITY_CUTOFF + 0.05, ax.get_ylim()[1] * 0.95, f"activity cut-off\npIC50 = {ACTIVITY_CUTOFF}", va="top")
ax.set(xlabel="Measured pIC50", ylabel="Number of compounds")
ax.set_title(f"ChEMBL DRD2 training set: {len(training)} compounds with measured IC50")
ax.legend(loc="upper left")
save(fig, "01_training_set_pIC50_distribution.png")

## Figures 02 and 03 – Model × fingerprint benchmarks (5-fold cross-validation)

In [ ]:
FINGERPRINTS = {"maccs": "MACCS", "morgan2": "Morgan r=2", "morgan3": "Morgan r=3"}


def benchmark_plot(results, models, metrics, title, footnote, filename):
    """Grouped bar chart of cross-validation mean ± SD for each model and fingerprint.

    metrics: list of (column prefix, y-axis label, y-limits)
    """
    fig, axes = plt.subplots(1, len(metrics), figsize=(13, 6))
    x = np.arange(len(FINGERPRINTS))
    width = 0.26
    for ax, (metric, ylabel, ylim) in zip(axes, metrics):
        for i, (model, color) in enumerate(zip(models, [BLUE, ORANGE, GREEN])):
            rows = results[results["model"] == model].set_index("fingerprint").loc[list(FINGERPRINTS)]
            means, stds = rows[f"{metric}_mean"], rows[f"{metric}_std"]
            positions = x + (i - 1) * width
            ax.bar(positions, means, width * 0.9, yerr=stds, capsize=4, color=color, label=model,
                   error_kw={"ecolor": "#444444", "lw": 1})
            for pos, mean, std in zip(positions, means, stds):
                ax.text(pos, max(mean, 0) + std + 0.012 * (ylim[1] - ylim[0]), f"{mean:.2f}",
                        ha="center", va="bottom", fontsize=10)
        ax.set(xticks=x, xticklabels=FINGERPRINTS.values(), ylabel=ylabel, ylim=ylim)
        ax.grid(axis="x", visible=False)
    axes[0].legend(ncol=3, loc="lower left", bbox_to_anchor=(0, 1.04))
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold", fontsize=17)
    fig.text(0.01, -0.02, footnote, color="#555555", fontsize=11)
    fig.tight_layout()
    save(fig, filename)

In [ ]:
classification = pd.read_csv(PROJECT / "results" / "ml_results_crossvalidation.csv")
benchmark_plot(
    classification,
    models=["RF", "SVM", "ANN"],
    metrics=[("auc", "ROC-AUC (5-fold CV)", (0.5, 1.08)), ("accuracy", "Accuracy (5-fold CV)", (0.5, 1.08))],
    title=f"Classification (active: pIC50 ≥ {ACTIVITY_CUTOFF}): model × fingerprint benchmark",
    footnote="Bars: mean of 5-fold cross-validation; error bars: standard deviation.",
    filename="02_classification_benchmark.png",
)

In [ ]:
regression = pd.read_csv(PROJECT / "results" / "reg_results_crossvalidation.csv")
benchmark_plot(
    regression,
    models=["RF", "SVR", "ANN"],
    metrics=[("r2", "R² (5-fold CV)", (0, 0.75)), ("rmse", "RMSE, pIC50 units (5-fold CV)", (0, 1.3))],
    title="Regression QSAR (predicting pIC50): model × fingerprint benchmark",
    footnote="Bars: mean of 5-fold cross-validation; error bars: standard deviation. "
             "SVR / Morgan r=2 was used to score the screening library.",
    filename="03_regression_benchmark.png",
)

## Figure 04 – QSAR model: cross-validated predictions

Out-of-fold predictions of the final model (SVR, RBF kernel, Morgan r=2, 2048 bits) from shuffled 5-fold CV.
`random_state=42` reproduces the presented figure (R² = 0.48, RMSE = 0.78).

In [ ]:
morgan2 = pd.read_csv(PROJECT / "data" / "DopamineD2_morgan2.csv")
X = morgan2.filter(like="bit_").values
y = morgan2["pIC50"].values

y_cv = cross_val_predict(
    SVR(kernel="rbf", C=1.0, gamma="scale"), X, y, cv=KFold(n_splits=5, shuffle=True, random_state=42)
)
r2, rmse = r2_score(y, y_cv), mean_squared_error(y, y_cv) ** 0.5

lims = (3.5, 10.2)
line = np.array(lims)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.fill_between(line, line - 1, line + 1, color="#EFEFED", lw=0, label="± 1 log unit")
ax.plot(line, line, ls="--", color="#444444")
ax.scatter(y, y_cv, s=45, color=BLUE, alpha=0.8, edgecolor="white", lw=0.6)
ax.text(0.97, 0.06, f"R² = {r2:.2f}\nRMSE = {rmse:.2f}\nn = {len(y)}", transform=ax.transAxes,
        ha="right", va="bottom", fontsize=15)
ax.set(xlim=lims, ylim=lims, aspect="equal", xlabel="Measured pIC50", ylabel="Predicted pIC50 (cross-validated)")
ax.set_title("QSAR model: SVR on Morgan r=2 fingerprints", loc="center")
ax.legend(loc="upper left", fontsize=14)
save(fig, "04_QSAR_predicted_vs_measured.png")

## Figure 05 – Virtual screening: predicted pIC50 of both libraries

In [ ]:
random_library = pd.read_csv(PROJECT / "results" / "random_smiles_predicted_pic50.csv")
scaffold_library = pd.read_csv(DATA / "Similarity_Scaffold_predicted_pIC50.csv")

distributions = [
    (random_library["predicted_pIC50"], GREY, f"Random ZINC library (n = {len(random_library):,})"),
    (scaffold_library["pred_pIC50"], ORANGE, f"Scaffold-similarity library (n = {len(scaffold_library):,})"),
    (training["pIC50"], BLUE, f"Training set, measured (n = {len(training)})"),
]
bins = np.arange(3.5, 10.26, 0.15)

fig, ax = plt.subplots(figsize=(13, 5.5))
for values, color, label in distributions:
    ax.hist(values, bins=bins, density=True, histtype="stepfilled", color=color, alpha=0.35)
    ax.hist(values, bins=bins, density=True, histtype="step", color=color, lw=2, label=label)
ax.axvline(ACTIVITY_CUTOFF, color="black", ls="--", lw=1)
ax.set(xlabel="pIC50 (predicted for libraries, measured for training set)", ylabel="Density")
ax.set_title("Virtual screening: scaffold-based library scores higher than random ZINC compounds")
ax.legend(loc="upper right")
save(fig, "05_virtual_screening_predicted_pIC50.png")

## Figure 06 – Top-100 ML-ranked compounds re-scored by docking

In [ ]:
flagged = top100[["PAINS", "BRENK"]].notna().any(axis=1)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.scatter(top100.loc[flagged, "pred_pIC50"], top100.loc[flagged, "vina"], marker="X", s=90,
           color=GREY, label="Flagged by PAINS/Brenk filter")
ax.scatter(top100.loc[~flagged, "pred_pIC50"], top100.loc[~flagged, "vina"], s=55, color=BLUE,
           alpha=0.8, edgecolor="white", label="Passed substructure filters")
ax.scatter(hits["pred_pIC50"], hits["vina_top100"], s=320, facecolor="none", edgecolor=ORANGE, lw=2.5,
           label="Selected top-5 hits")
for _, hit in hits.iterrows():
    ax.annotate(hit["label"], (hit["pred_pIC50"], hit["vina_top100"]), xytext=(12, 0),
                textcoords="offset points", va="center", fontweight="bold")
ax.invert_yaxis()
ax.set(xlabel="Predicted pIC50 (SVR QSAR)", ylabel="Vina score, 6CM4 (kcal/mol)\nmore negative = better")
ax.set_title("Top-100 ML-ranked compounds re-scored by docking")
ax.legend(loc="lower right")
save(fig, "06_ML_vs_docking_top100.png")

## Figure 07 – Docking validation: reference ligands vs. screened compounds

Control ligands were docked into 6CM4 with the same box and settings; the score of each top-ranked pose is
read from its Vina output.

In [ ]:
CONTROLS = {  # label: run directory in docking/runs
    "Risperidone (redock)": "Risperidone",
    "Approved drug CHEMBL2016623": "Approved_Dru0g_CHEMBL2016623",
    "CHEMBL567 (pIC50 9.5)": "ligand",
    "Dopamine": "Dopamine",
    "Ulotaront (non-binder)": "Non_binder_fixed",
    "Ethane (negative ctrl)": "poor_candidate",
}
control_scores = {label: best_vina_score(RUNS / run / "docked_poses.pdbqt") for label, run in CONTROLS.items()}

random_top100 = pd.read_csv(PROJECT / "results" / "random_smiles_top100_docking" / "top_100_docking_summary.csv")
groups = [
    ("Random ZINC\ntop-100", random_top100["vina_score_kcal_mol"], GREY),
    ("Scaffold library\ntop-100", top100["vina"], BLUE),
    ("Selected\ntop-5 hits", hits["vina_score_6CM4_kcal_mol"], ORANGE),
]

rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(11.5, 5.8))
for i, (_, scores, color) in enumerate(groups):
    ax.boxplot(scores, positions=[i], widths=0.5, showfliers=False,
               medianprops={"color": "black", "lw": 2.5}, whiskerprops={"color": "#444444"},
               capprops={"color": "#444444"}, boxprops={"color": "#444444"})
    ax.scatter(i + rng.uniform(-0.18, 0.18, len(scores)), scores, s=40, color=color, edgecolor="white", zorder=3)
    ax.text(i, -12.5, f"median {scores.median():.1f}", ha="center", color="#555555")
for label, score in control_scores.items():
    ax.axhline(score, color="#555555", ls=":", lw=1.2)
    ax.text(3.65, score, label, ha="right", va="bottom", fontsize=11)
ax.set_xticks(range(len(groups)), [name for name, _, _ in groups])
ax.set_xlim(-0.5, 3.7)
ax.set_ylim(-1.5, -12.8)  # inverted: better scores at the top
ax.grid(axis="x", visible=False)
ax.set_ylabel("Vina score, 6CM4 (kcal/mol)\nmore negative = better")
ax.set_title("Docking validation: reference ligands vs screened compounds")
save(fig, "07_docking_scores_vs_controls.png")

## Figure 08 – Inactive (6CM4) vs. active (9BS9) state docking

Δ = Vina(9BS9) − Vina(6CM4); a positive Δ means the compound prefers the inactive state.

In [ ]:
dopamine_dir = RUNS / "state_comparisons" / "Dopamine"
states = pd.concat([
    hits[["label", "vina_score_6CM4_kcal_mol", "vina_score_9BS9_kcal_mol"]],
    pd.DataFrame({
        "label": ["Dopamine"],
        "vina_score_6CM4_kcal_mol": [best_vina_score(dopamine_dir / "6CM4" / "docked_poses.pdbqt")],
        "vina_score_9BS9_kcal_mol": [best_vina_score(dopamine_dir / "9BS9" / "docked_poses.pdbqt")],
    }),
], ignore_index=True)
states["delta"] = states["vina_score_9BS9_kcal_mol"] - states["vina_score_6CM4_kcal_mol"]

fig, ax = plt.subplots(figsize=(11.5, 5.8))
for i, row in states.iterrows():
    ax.plot([row["vina_score_6CM4_kcal_mol"], row["vina_score_9BS9_kcal_mol"]], [i, i], color=GREY, lw=3, zorder=1)
    ax.text(row["vina_score_9BS9_kcal_mol"] + 0.25, i, f"Δ = {row['delta']:+.2f}", ha="right", va="center",
            color="#444444")
ax.scatter(states["vina_score_6CM4_kcal_mol"], states.index, s=180, color=BLUE, edgecolor="white", zorder=2,
           label="6CM4 – inactive state (risperidone-bound, X-ray)")
ax.scatter(states["vina_score_9BS9_kcal_mol"], states.index, s=180, color=ORANGE, edgecolor="white", zorder=3,
           label="9BS9 – active state (LSD/mini-Gαo, cryo-EM)")
ax.set_yticks(states.index, states["label"])
ax.set_ylim(len(states) - 0.4, -1.6)
ax.set_xlim(-5.5, -12.3)
ax.set_xlabel("Vina score (kcal/mol)   more negative = better →")
ax.set_title("All hits dock better to the inactive state – consistent with antagonism")
ax.legend(loc="upper left")
save(fig, "08_active_vs_inactive_state_docking.png")

## Figure 09 – Structures of the top-5 hits

In [ ]:
options = Draw.MolDrawOptions()
options.legendFontSize = 30
legends = [
    f"{h['label']}   pred. pIC50 {h['pred_pIC50']:.2f}  |  Vina {h['vina_score_6CM4_kcal_mol']:.1f} kcal/mol"
    for _, h in hits.iterrows()
]
grid = Draw.MolsToGridImage(
    [Chem.MolFromSmiles(s) for s in hits["smiles"]],
    molsPerRow=3,
    subImgSize=(1000, 700),
    legends=legends,
    drawOptions=options,
    returnPNG=False,
)
grid.save(FIG_DIR / "09_top5_hits_structures.png")
grid

## Figure 10 – Cluster 4: docking fails for flexible bivalent ligands

In [ ]:
cluster4 = pd.read_csv(DATA / "cluster4_docking_results.csv")
r, _ = pearsonr(cluster4["pIC50"], cluster4["vina_score"])

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(cluster4["pIC50"], cluster4["vina_score"], s=110, color=BLUE, edgecolor="white")
ax.invert_yaxis()
ax.text(0.03, 0.04, f"Pearson r = {r:.2f}, n = {len(cluster4)}\nmany rotatable bonds → poor docking",
        transform=ax.transAxes, color="#555555")
ax.set(xlabel="Measured pIC50", ylabel="Vina score, 6CM4 (kcal/mol) ← better")
ax.set_title("Cluster 4 (flexible bivalent ligands)")
save(fig, "10_cluster4_docking_vs_pIC50.png")

## Figure 11 – Docking reproducibility across random seeds (Hit 01)

In [ ]:
seeds = pd.read_csv(RUNS / "ligand_1_supplied_6CM4" / "docking_summary.csv")
scores = seeds["best_vina_score_kcal_mol"]

fig, ax = plt.subplots(figsize=(8, 6))
x = np.arange(len(seeds))
ax.scatter(x, scores, s=160, color=BLUE, edgecolor="white", zorder=3)
ax.axhline(scores.mean(), color="#444444", ls="--")
for xi, score in zip(x, scores):
    ax.text(xi, score - 0.1, f"{score:.2f}", ha="center", va="bottom", fontsize=13)
ax.text(0.02, 0.04,
        f"mean {scores.mean():.2f} ± {scores.std():.2f} kcal/mol "
        f"(n = {len(seeds)}, exhaustiveness {seeds['exhaustiveness'].iloc[0]})",
        transform=ax.transAxes, color="#555555", fontsize=12)
ax.set_xticks(x, seeds["search_seed"].astype(str))
ax.set_ylim(-9.5, -11.5)
ax.grid(axis="x", visible=False)
ax.set(xlabel="Random seed", ylabel="Best Vina score (kcal/mol)")
ax.set_title("Hit 01: docking is reproducible across seeds")
save(fig, "11_docking_seed_reproducibility.png")

## Figure 12 – MD simulation of Hit 01 in 6CM4 (+ Na⁺)

Same post-processing as Workflow step 9: OpenMM wraps each chain/molecule into the box separately, so we
re-centre on the receptor before computing the RMSD. The Asp114 distance is the shortest distance between the
Asp114 carboxylate oxygens and the protonated piperazine nitrogen (N11), with the minimum-image convention.

In [ ]:
import MDAnalysis as mda
from MDAnalysis import transformations
from MDAnalysis.analysis import rms
from MDAnalysis.lib.distances import distance_array

universe = mda.Universe(str(DATA / "topology.pdb"), str(DATA / "trajectory.xtc"))
receptor_chain1 = universe.select_atoms("protein and resid 35:222")
receptor = universe.select_atoms("protein")
universe.trajectory.add_transformations(
    transformations.center_in_box(receptor_chain1),
    transformations.wrap(universe.atoms, compound="residues"),
    transformations.center_in_box(receptor),
    transformations.wrap(universe.atoms, compound="residues"),
)

# RMSD of the backbone, and of the ligand heavy atoms after fitting on the backbone
rmsd_analysis = rms.RMSD(universe, universe, select="backbone", groupselections=["resname LIG and not name H*"], ref_frame=0)
rmsd_analysis.run()
md_results = pd.DataFrame(rmsd_analysis.results.rmsd[:, 1:], columns=["time_ps", "backbone", "ligand"])

asp114_oxygens = universe.select_atoms("resid 114 and resname ASP and name OD1 OD2")
piperazine_n = universe.select_atoms("resname LIG and name N11")
md_results["asp114"] = [
    distance_array(asp114_oxygens.positions, piperazine_n.positions, box=universe.dimensions).min()
    for _ in universe.trajectory
]
md_results["time_ns"] = md_results["time_ps"] / 1000

md_log = pd.read_csv(DATA / "simulation_log.txt", sep="\t")
md_log["time_ns"] = md_log["Step"] * 2e-6  # 2 fs time step

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].plot(md_results["time_ns"], md_results["backbone"], color=BLUE, lw=2, label="Protein backbone")
axes[0].plot(md_results["time_ns"], md_results["ligand"], color=ORANGE, lw=2, label="Ligand (heavy atoms)")
axes[0].set(ylabel="RMSD vs. frame 0 (Å)", ylim=(0, None), title="RMSD")
axes[0].legend(loc="lower right", fontsize=12)

axes[1].axhspan(0, 4, color=GREEN, alpha=0.12, lw=0)
axes[1].text(0.0, 3.7, "salt bridge ≤ 4 Å", va="top", color="#555555", fontsize=12)
axes[1].plot(md_results["time_ns"], md_results["asp114"], color=GREEN, lw=2)
axes[1].set(ylabel="Asp114 O – piperazine N⁺ (Å)", ylim=(0, 8), title="Asp114 anchor")

axes[2].plot(md_log["time_ns"], md_log["Temperature (K)"], color=GREY, lw=2)
axes[2].set(ylabel="Temperature (K)", ylim=(290, 310), title="Thermostat (300 K)")

for ax in axes:
    ax.set_xlabel("Time (ns)")
fig.suptitle("MD of Hit 01 in inactive DRD2 (6CM4 + Na⁺), 1 ns, Amber14/OpenFF 2.2, TIP3P",
             x=0.01, ha="left", fontweight="bold", fontsize=17)
fig.tight_layout()
save(fig, "12_MD_Hit01_RMSD_Asp114.png")

print(f"Asp114 distance: mean {md_results['asp114'].mean():.2f} Å, "
      f"{(md_results['asp114'] <= 4).mean():.0%} of frames ≤ 4 Å")

## PyMOL helpers for the 3D figures (13–16)

Every structure is superposed onto the prepared 6CM4 receptor (Cα atoms), so all pocket views share one
camera. `cmd.align` only moves the receptor object, so the same transformation is copied to the ligands.
The PyMOL renders are combined with titles, captions and legends in matplotlib.

In [ ]:
from pymol import cmd, util

REFERENCE_RECEPTOR = PROJECT / "docking" / "prepared" / "6CM4" / "drd2_repaired.pdb"
KEY_RESIDUES = (100, 114, 118, 184, 193, 197, 386, 389, 390, 393, 416)
ONE_LETTER = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q", "GLU": "E", "GLY": "G",
    "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P", "SER": "S",
    "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
}
RENDER_DIR = FIG_DIR / "3D"
RENDER_DIR.mkdir(exist_ok=True)


def hex_to_rgb(color):
    return [int(color[i:i + 2], 16) / 255 for i in (1, 3, 5)]


def load_first_pose(path, name):
    """Load only the top-ranked pose (state 1) of a docking output, without hydrogens."""
    cmd.load(str(path), f"{name}_all")
    cmd.create(name, f"{name}_all", 1, 1)
    cmd.delete(f"{name}_all")
    cmd.remove(f"{name} and hydro")


def superpose_on_reference(receptor, ligands):
    """Superpose the receptor object on 6CM4 and move the ligand objects along with it."""
    cmd.load(str(REFERENCE_RECEPTOR), "reference")
    cmd.align(f"{receptor} and polymer.protein and name CA", "reference and name CA")
    for ligand in ligands:
        cmd.matrix_copy(receptor, ligand)
    cmd.delete("reference")


def asp114_contact(receptor, ligand):
    """Closest Asp114 carboxylate oxygen – ligand nitrogen pair: (distance, oxygen name, nitrogen ID)."""
    pairs = [
        (cmd.get_distance(f"{receptor} and resi 114 and name {oxygen}", f"{ligand} and id {atom.id}"), oxygen, atom.id)
        for oxygen in ("OD1", "OD2")
        for atom in cmd.get_model(f"{ligand} and elem N").atom
    ]
    return min(pairs)


def style_pocket(receptor, ligand_colors, cartoon_color="grey90", labels=KEY_RESIDUES):
    """Transparent cartoon, pocket residues (≤ 4 Å) as grey sticks and ligands as thick coloured sticks."""
    ligands = " or ".join(ligand_colors)
    cmd.hide("everything")
    cmd.bg_color("white")
    for setting, value in [("ray_opaque_background", 1), ("orthoscopic", 1), ("depth_cue", 0),
                           ("ray_shadows", 0), ("specular", 0.2), ("antialias", 2),
                           ("cartoon_transparency", 0.65), ("stick_radius", 0.17), ("label_size", 22),
                           ("label_color", "black"), ("label_font_id", 5), ("float_labels", 1),
                           ("dash_radius", 0.06), ("dash_gap", 0.25)]:
        cmd.set(setting, value)

    cmd.set_color("contact_color", hex_to_rgb("#E6A100"))
    cmd.set("dash_color", "contact_color")
    cmd.set_color("cartoon_color", hex_to_rgb(cartoon_color) if cartoon_color.startswith("#") else cmd.get_color_tuple(cartoon_color))
    cmd.show("cartoon", receptor)
    cmd.color("cartoon_color", receptor)

    cmd.select("pocket", f"byres ({receptor} and polymer.protein within 4 of ({ligands}))")
    cmd.show("sticks", "pocket and not name N+C+O")
    cmd.color("grey70", "pocket and elem C")
    util.cnc("pocket")

    for ligand, color in ligand_colors.items():
        cmd.set_color(f"{ligand}_color", hex_to_rgb(color))
        cmd.show("sticks", ligand)
        cmd.color(f"{ligand}_color", f"{ligand} and elem C")
        cmd.set("stick_radius", 0.28, ligand)
    util.cnc(ligands)

    for resi in labels:
        selection = f"pocket and resi {resi} and name CA"
        if cmd.count_atoms(selection):
            resn = cmd.get_model(selection).atom[0].resn
            cmd.label(selection, f'"{ONE_LETTER[resn]}{resi}"')
    cmd.deselect()


def render(path, width=2000, height=1400):
    cmd.png(str(path), width=width, height=height, dpi=300, ray=1)
    return plt.imread(path)


def show_image(ax, image):
    ax.imshow(image)
    ax.axis("off")


# One camera for all pocket views: centred on Hit 01 in 6CM4
cmd.reinitialize()
load_first_pose(HIT_DIR / "01_Hit_01" / "6CM4" / "docked_poses.pdbqt", "camera_ligand")
cmd.load(str(REFERENCE_RECEPTOR), "camera_receptor")
cmd.orient("camera_ligand")
cmd.zoom("camera_ligand", 6)
cmd.turn("y", 180)  # look at the pocket with TM6/7 on the left, as in the slides
POCKET_VIEW = cmd.get_view()

## Figure 13 – Docked pose of each hit in the inactive (6CM4) and active (9BS9) state

Orange dashes: the Asp114 salt bridge (labelled with its distance) and other polar contacts ≤ 3.5 Å.

In [ ]:
STATE_PANELS = [
    ("6CM4", "Inactive state – 6CM4", BLUE, "#EDF3FB"),
    ("9BS9", "Active state – 9BS9", ORANGE, "#FDF0EA"),
]


def render_hit_pose(hit_dir, state, cartoon_color, png):
    """Render one docked hit in its receptor; return the Asp114–N distance."""
    cmd.reinitialize()
    cmd.load(str(hit_dir / state / "drd2_repaired.pdb"), "receptor")
    cmd.remove("receptor and (hydro or solvent)")
    load_first_pose(hit_dir / state / "docked_poses.pdbqt", "ligand")
    superpose_on_reference("receptor", ["ligand"])
    style_pocket("receptor", {"ligand": ORANGE}, cartoon_color=cartoon_color)

    distance, oxygen, nitrogen = asp114_contact("receptor", "ligand")
    cmd.distance("salt_bridge", f"receptor and resi 114 and name {oxygen}", f"ligand and id {nitrogen}")
    cmd.distance("polar", "ligand", "pocket", 3.5, mode=2)
    cmd.hide("labels", "polar")
    cmd.set("label_size", 22, "salt_bridge")

    cmd.set_view(POCKET_VIEW)
    render(png)
    return distance


legend_handles = [
    Line2D([], [], color=ORANGE, lw=6, label="Docked ligand"),
    Line2D([], [], color="#E6A100", lw=3, ls="--", label="Asp114 salt bridge (labelled, Å) / polar contacts ≤ 3.5 Å"),
    Line2D([], [], color="#C8C8C8", lw=6, label="Pocket residues ≤ 4 Å"),
]

for i, hit in hits.iterrows():
    hit_dir = HIT_DIR / hit["run_directory_6CM4"].split("/")[0]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6.3))
    for ax, (state, title, color, cartoon) in zip(axes, STATE_PANELS):
        png = RENDER_DIR / f"R10_{i + 1:02d}_{hit['compound_id']}_{state}_pose.png"
        distance = render_hit_pose(hit_dir, state, cartoon, png)
        show_image(ax, plt.imread(png))
        ax.set_title(title, color=color, fontsize=17)
        ax.text(0, -0.04, f"Vina {hit[f'vina_score_{state}_kcal_mol']:.1f} kcal/mol  ·  Asp114–N {distance:.1f} Å",
                transform=ax.transAxes, va="top", color="#555555", fontsize=14)
    fig.suptitle(f"{hit['label']} – docked pose (predicted pIC50 {hit['pred_pIC50']:.2f})",
                 x=0.01, ha="left", fontweight="bold", fontsize=20)
    fig.legend(handles=legend_handles, loc="lower center", ncol=3, fontsize=13, bbox_to_anchor=(0.5, -0.06))
    fig.tight_layout()
    save(fig, f"13_{i + 1:02d}_{hit['compound_id']}_pose_inactive_vs_active.png")

## Figure 14 – Redocking validation: risperidone in 6CM4

The crystal ligand (8NU) gets its bond orders from the risperidone SMILES; the RMSD between the top Vina pose and
the crystal pose is computed in place (no re-alignment) and is symmetry-corrected by RDKit.

In [ ]:
risperidone_dir = RUNS / "Risperidone"
risperidone_smiles = (PROJECT / "docking" / "examples" / "drd2" / "Risperidone.smi").read_text().split()[0]

cmd.reinitialize()
cmd.load(str(risperidone_dir / "6CM4.cif"), "receptor")
cmd.extract("crystal", "receptor and resn 8NU")
cmd.remove("(receptor or crystal) and hydro")
cmd.remove("receptor and not polymer.protein")
load_first_pose(risperidone_dir / "docked_poses.sdf", "docked")

crystal_mol = AllChem.AssignBondOrdersFromTemplate(
    Chem.MolFromSmiles(risperidone_smiles), Chem.MolFromPDBBlock(cmd.get_pdbstr("crystal"))
)
docked_mol = Chem.RemoveHs(next(Chem.SDMolSupplier(str(risperidone_dir / "docked_poses.sdf"))))
redock_rmsd = rdMolAlign.CalcRMS(docked_mol, crystal_mol)
redock_score = best_vina_score(risperidone_dir / "docked_poses.pdbqt")

superpose_on_reference("receptor", ["crystal", "docked"])
style_pocket("receptor", {"crystal": BLUE, "docked": ORANGE})
cmd.set("stick_transparency", 0.25, "crystal")
cmd.set_view(POCKET_VIEW)
image = render(RENDER_DIR / "R02_redock_risperidone_crystal_vs_docked.png")

fig, ax = plt.subplots(figsize=(10, 7.5))
show_image(ax, image)
ax.set_title("Redocking validation: risperidone in 6CM4", fontsize=20)
ax.legend(handles=[Line2D([], [], color=BLUE, lw=6, label="Crystal pose (6CM4, ligand 8NU)"),
                   Line2D([], [], color=ORANGE, lw=6, label="Vina top-ranked pose")],
          loc="upper right", frameon=True, fontsize=14)
ax.text(0, -0.02, f"Heavy-atom RMSD docked vs. crystal = {redock_rmsd:.2f} Å (symmetry-corrected)  ·  "
                  f"Vina {redock_score:.2f} kcal/mol",
        transform=ax.transAxes, va="top", color="#555555", fontsize=15)
save(fig, "14_redock_risperidone_validation.png")

## Figure 15 – Top-5 hits overlaid in 6CM4

In [ ]:
cmd.reinitialize()
cmd.load(str(REFERENCE_RECEPTOR), "receptor")
cmd.remove("receptor and (hydro or solvent)")
hit_colors = {}
for i, hit in hits.iterrows():
    name = f"hit{i + 1:02d}"
    load_first_pose(HIT_DIR / hit["run_directory_6CM4"] / "docked_poses.pdbqt", name)
    hit_colors[name] = HIT_COLORS[i]
superpose_on_reference("receptor", list(hit_colors))
style_pocket("receptor", hit_colors)
cmd.set("stick_radius", 0.22, " or ".join(hit_colors))
cmd.set_view(POCKET_VIEW)
image = render(RENDER_DIR / "R03_all_hits_overlay_6CM4.png")

fig, ax = plt.subplots(figsize=(10, 7.5))
show_image(ax, image)
ax.set_title("Top-5 hits share one binding mode in 6CM4", fontsize=20)
ax.legend(handles=[Line2D([], [], color=c, lw=6, label=l) for c, l in zip(HIT_COLORS, hits["label"])],
          loc="upper right", frameon=True, fontsize=15)
ax.text(0, -0.02, "Basic piperazine N anchored by Asp114; aryl group in the deep orthosteric pocket",
        transform=ax.transAxes, va="top", color="#555555", fontsize=15)
save(fig, "15_top5_hits_overlay.png")

## Figure 16 – DRD2 overview: Hit 01 and the allosteric Na⁺ ion

The membrane normal is put along the vertical axis (extracellular side up) using the principal axes of the
7TM bundle.

In [ ]:
cmd.reinitialize()
cmd.load(str(PROJECT.parent / "Naked_6CM4_NA_lig1.mmcif"), "receptor")
cmd.extract("ligand", "receptor and resn LIG")
cmd.extract("sodium", "receptor and resn NA")
cmd.remove("receptor and not polymer.protein")
cmd.remove("hydro")

cmd.hide("everything")
cmd.bg_color("white")
for setting, value in [("ray_opaque_background", 1), ("orthoscopic", 1), ("depth_cue", 0), ("antialias", 2),
                       ("ray_shadows", 0), ("specular", 0.3), ("cartoon_transparency", 0.0)]:
    cmd.set(setting, value)
cmd.set_color("overview_cartoon", hex_to_rgb("#C9D6EA"))
cmd.set_color("hit_carbon", hex_to_rgb(ORANGE))
cmd.set_color("sodium_color", hex_to_rgb("#5B4BD6"))
cmd.show("cartoon", "receptor")
cmd.color("overview_cartoon", "receptor")
cmd.show("spheres", "ligand")
cmd.color("hit_carbon", "ligand and elem C")
util.cnc("ligand")
cmd.show("spheres", "sodium")
cmd.color("sodium_color", "sodium")

# Orient so that the bundle axis is vertical and the ligand (extracellular side) is at the top
cmd.orient("receptor")
cmd.turn("z", 90)
view = cmd.get_view()
if np.dot(cmd.get_coords("ligand").mean(axis=0) - cmd.get_coords("sodium").mean(axis=0),
          np.array(view[:9]).reshape(3, 3)[:, 1]) < 0:
    cmd.turn("z", 180)
cmd.zoom("receptor", 2)
image = render(RENDER_DIR / "R01_DRD2_overview_Hit01_Na.png", width=1500, height=2000)

fig, ax = plt.subplots(figsize=(7.5, 10))
show_image(ax, image)
ax.set_title("DRD2 (6CM4) with Hit 01", fontsize=22)
ax.text(1, 0.99, "extracellular", transform=ax.transAxes, ha="right", va="top", color="#555555", fontsize=17)
ax.text(1, 0.01, "intracellular", transform=ax.transAxes, ha="right", va="bottom", color="#555555", fontsize=17)
ax.legend(handles=[Patch(color=ORANGE, label="Hit 01 (docked)"), Patch(color="#5B4BD6", label="Na⁺ (allosteric site)")],
          loc="lower left", frameon=True, fontsize=17)
save(fig, "16_DRD2_overview.png")

## Figure 17 – Residue–ligand distances during the MD simulation

Per-residue shortest heavy-atom distance to Hit 01 over the trajectory (from `MD_residue_ligand_distances.csv`).
Only residues with a mean distance ≤ 4.5 Å are shown; residues in contact in ≥ 90 % of the frames are in bold.
Ballesteros–Weinstein numbers are added for the TM residues.

In [ ]:
BALLESTEROS_WEINSTEIN = {
    "Trp100": "ECL1", "Asp114": "3.32", "Val115": "3.33", "Cys118": "3.36", "Thr119": "3.37", "Ala122": "3.40",
    "Ile184": "ECL2", "Phe189": "5.38", "Ser193": "5.42", "Ser197": "5.46", "Phe198": "5.47", "Trp386": "6.48",
    "Phe389": "6.51", "Phe390": "6.52", "His393": "6.55", "Tyr408": "7.35", "Thr412": "7.39", "Tyr416": "7.43",
}

contacts = pd.read_csv(PROJECT / "MD_residue_ligand_distances.csv")
contacts.columns = ["residue", "start", "mean", "sd", "min", "max", "end", "contact_pct", "closest_pair"]
contacts = contacts[contacts["mean"] <= 4.5]

columns = ["Residue", "BW", "Mean ± SD (Å)", "Range (Å)", "Contact (%)", "Closest atoms"]
rows = [
    [c.residue, BALLESTEROS_WEINSTEIN.get(c.residue, ""), f"{c.mean:.2f} ± {c.sd:.2f}", f"{c.min:.2f} – {c.max:.2f}",
     f"{c.contact_pct:.0f}", c.closest_pair.replace("–", " ··· ")]
    for c in contacts.itertuples()
]

fig, ax = plt.subplots(figsize=(11, 0.32 * len(rows) + 0.8))
ax.axis("off")
table = ax.table(cellText=rows, colLabels=columns, loc="center", cellLoc="center",
                 colWidths=[0.15, 0.09, 0.2, 0.18, 0.15, 0.21])
table.auto_set_font_size(False)
table.set_fontsize(13)
table.scale(1, 1.6)
for (row, col), cell in table.get_celld().items():
    cell.set_edgecolor("white")
    cell.set_linewidth(2)
    if col in (0, 5):
        cell.set_text_props(ha="left")
        cell.PAD = 0.03
    if row == 0:
        cell.set_text_props(fontweight="bold")
        continue
    if row % 2 == 0:
        cell.set_facecolor("#F2F3F7")
    if contacts["contact_pct"].iloc[row - 1] >= 90 and col in (0, 2, 4):
        cell.set_text_props(fontweight="bold")
save(fig, "17_MD_residue_ligand_distances.png")